## Setup & Init

In [1]:
from IPython.display import Markdown

In [2]:
import langchain_hana
print(langchain_hana.__version__)

1.0.2


## Connect to SAP HANA Cloud DB instance

First, create a connection to your SAP HANA Cloud instance.

In [3]:
import os

from dotenv import load_dotenv
from hdbcli import dbapi

# Load environment variables if needed
load_dotenv()

# Establish connection to SAP HANA Cloud
connection = dbapi.connect(
    address=os.environ.get("HANADB_URL"),
    port=os.environ.get("HANADB_PRT"),
    user=os.environ.get("HANADB_USR"),
    password=os.environ.get("HANADB_PWD"),
    autocommit=True,
    sslValidateCertificate=False,
)

## Initialize the `HanaRdfGraph`

To power the QA chain, you first need a `HanaRdfGraph` instance that:

1. Loads your ontology schema (in Turtle)  
2. Executes SPARQL queries against your SAP HANA Cloud data graph

## Question Answering over a Knowledge Graph

### Instantiate the `HanaRdfGraph` pointing at our data graph

In [4]:
from langchain_hana import HanaRdfGraph

In [5]:
# Set up the Knowledge Graph with explicite ontology
graph_uri = "teched2025_devkeynote"

graph = HanaRdfGraph(
    connection=connection, graph_uri=graph_uri, ontology_local_file="./ontology.ttl", ontology_local_file_format="turtle"
)

In [6]:
# Set up the Knowledge Graph
graph_uri = "teched2025_devkeynote"

graph = HanaRdfGraph(
    connection=connection, graph_uri=graph_uri, auto_extract_ontology=True
)

In [7]:
print(graph.get_schema.serialize(format="turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://schema.org/about> a owl:ObjectProperty ;
    rdfs:label "about" ;
    rdfs:domain <http://schema.org/Event> ;
    rdfs:range <http://schema.org/Service> .

<http://schema.org/addressCountry> a owl:ObjectProperty ;
    rdfs:label "addressCountry" ;
    rdfs:domain <http://schema.org/Person> ;
    rdfs:range xsd:uri .

<http://schema.org/jobTitle> a owl:DatatypeProperty ;
    rdfs:label "jobTitle" ;
    rdfs:domain <http://schema.org/Person> ;
    rdfs:range xsd:string .

<http://schema.org/name> a owl:DatatypeProperty ;
    rdfs:label "name" ;
    rdfs:domain <http://schema.org/Event>,
        <http://schema.org/Person>,
        <http://schema.org/Service>,
        <http://schema.org/SoftwareApplication> ;
    rdfs:range xsd:string .

<http://schema.org/performer> a owl:ObjectProperty ;
    rdfs:label "performer" ;
    rdfs:d

### Question Answering with `HanaSparqlQAChain`

`HanaSparqlQAChain` ties together:

1. **Schema-aware SPARQL generation**  
2. **Query execution** against SAP HANA  
3. **Natural-language answer formatting**


#### Initialization

You need:

- An **LLM** to generate and interpret queries  
- A **`HanaRdfGraph`** (with connection, `graph_uri`, and ontology)

### Connect to SAP AI Core instance in a given Cloud Foundry space

`cf login -a https://api.cf.eu10-005.hana.ondemand.com --origin a7rg4vxjp-platform -u codejamhanaai001@education.cloud.sap -p appgyver27-cap10`

In [ ]:
from cloudfoundry_client.client import CloudFoundryClient
import os

In [9]:
# use the default CF file, i.e. ~/.cf/config.json
client = CloudFoundryClient.build_from_cf_config(
    proxy = dict(http=os.environ.get('HTTP_PROXY', ''), https=os.environ.get('HTTPS_PROXY', ''))
)

In [10]:
my_aicore_serivce_instance=client.v3.service_instances.get_first(names=['default_aicore'])
display(my_aicore_serivce_instance['name'])

'default_aicore'

In [11]:
# Use a specific service key 'codejam-hanaai'
my_cf_servicebinding=client.v3.service_credential_bindings.get_first(names=['codejam-hanaai'])

In [12]:
display(my_cf_servicebinding)

{'guid': 'a694b4b5-5143-4976-a3d0-55f89f15274b',
 'created_at': '2026-04-09T15:13:13Z',
 'updated_at': '2026-04-09T15:13:13Z',
 'name': 'codejam-hanaai',
 'type': 'key',
 'last_operation': {'state': 'succeeded',
  'description': None,
  'type': 'create',
  'updated_at': '2026-04-09T15:13:13Z',
  'created_at': '2026-04-09T15:13:13Z'},
 'metadata': {'labels': {}, 'annotations': {}},
 'relationships': {'service_instance': {'data': {'guid': '2bfae4ab-2107-413a-a566-49fb4fccdca8'}}},
 'links': {'self': {'href': 'https://api.cf.eu10-005.hana.ondemand.com/v3/service_credential_bindings/a694b4b5-5143-4976-a3d0-55f89f15274b'},
  'details': {'href': 'https://api.cf.eu10-005.hana.ondemand.com/v3/service_credential_bindings/a694b4b5-5143-4976-a3d0-55f89f15274b/details'},
  'service_instance': {'href': 'https://api.cf.eu10-005.hana.ondemand.com/v3/service_instances/2bfae4ab-2107-413a-a566-49fb4fccdca8'},
  'parameters': {'href': 'https://api.cf.eu10-005.hana.ondemand.com/v3/service_credential_bindi

In [13]:
my_cf_servicebinding_details=my_cf_servicebinding.details()

In [14]:
my_cf_servicebinding_details

{'credentials': {'serviceurls': {'AI_API_URL': 'https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com'},
  'appname': '2bfae4ab-2107-413a-a566-49fb4fccdca8!b556622|aicore!b540',
  'clientid': 'sb-2bfae4ab-2107-413a-a566-49fb4fccdca8!b556622|aicore!b540',
  'clientsecret': 'a694b4b5-5143-4976-a3d0-55f89f15274b$_fKQTCAm_bUBFVpqeE-a9QR4q0r15YpeTF7jcJa_hVw=',
  'identityzone': 'hana-ai-codejam',
  'identityzoneid': '2962c53c-da55-4b6b-8677-8f56ed5f0501',
  'url': 'https://hana-ai-codejam.authentication.eu10.hana.ondemand.com',
  'credential-type': 'binding-secret'}}

In [15]:
# Load the AICore client
from ai_core_sdk.ai_core_v2_client import AICoreV2Client

# Create Connection
ai_core_client = AICoreV2Client(
    base_url      = my_cf_servicebinding_details['credentials']['serviceurls']['AI_API_URL'] + '/v2', # The present SAP AI Core API version is 2
    auth_url      = my_cf_servicebinding_details['credentials']['url'] + '/oauth/token', # Suffix to add
    client_id     = my_cf_servicebinding_details['credentials']['clientid'],
    client_secret = my_cf_servicebinding_details['credentials']['clientsecret'],
    resource_group='default'
)

#### Initialize the LLM

In [16]:
from langchain_hana import HanaSparqlQAChain
from gen_ai_hub.proxy.langchain.init_models import init_llm

In [17]:
from gen_ai_hub.proxy.gen_ai_hub_proxy.client import GenAIHubProxyClient

genai_client=GenAIHubProxyClient(ai_core_client=ai_core_client)

In [18]:
model_to_use=["gpt-4o", "gpt-4.1", "gpt-5", "gpt-5-mini", "anthropic--claude-4.5-sonnet", "gemini-2.5-pro"][0]

llm = init_llm(
    model_to_use,
    temperature=0.1,
    # model_id="anthropic.claude-sonnet-4-5-20250929-v1:0"
    proxy_client=genai_client
)

In [19]:
from langchain_core.prompts import PromptTemplate

### Read SPARQL_GENERATION_SELECT_TEMPLATE from file `sparql_generation_template__{model_to_use}.md`
with open(f"sparql_generation_template__{model_to_use}.md", "r") as f:
    SPARQL_GENERATION_SELECT_TEMPLATE = f.read()
#
SPARQL_GENERATION_SELECT_TEMPLATE = PromptTemplate(
    input_variables=["schema", "prompt"], template=SPARQL_GENERATION_SELECT_TEMPLATE
)

### Read SPARQL_QA_TEMPLATE from file `sparql_qa_template__{model_to_use}.md`
with open(f"sparql_qa_template__{model_to_use}.md", "r") as f:
    SPARQL_QA_TEMPLATE = f.read()
#
SPARQL_QA_PROMPT = PromptTemplate(
    input_variables=["context", "prompt"], template=SPARQL_QA_TEMPLATE
)


#### Create a SparQL QA Chain

In [20]:
from typing import cast

chain = HanaSparqlQAChain.from_llm(
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    graph=graph,
    # explicitly cast to PromptTemplate to satisfy the type checker
    sparql_generation_prompt=cast(PromptTemplate, SPARQL_GENERATION_SELECT_TEMPLATE),
    qa_prompt=cast(PromptTemplate, SPARQL_QA_PROMPT),
)

```Python
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from gen_ai_hub.proxy.langchain.init_models import init_llm

a = 2
b = 3

template_haha = """Question: {q} \n Answer should be as minimal as possible."""
prompt = PromptTemplate(template=template_haha, input_variables=['q'])

llm = init_llm(model_to_use)
chain = prompt | llm | StrOutputParser()
sum = chain.invoke({'q': f"Sum of {a} + {b}"})
display(Markdown(f"# {sum}"))
```

### Ask natural-language questions and print out the chain’s responses

In [21]:
prompt_from_user=[
    "What is the list of products to be shown?",
    "What is the data in the knowledge graph?",
    "What are the names of persons and countries they are in in the data? ",
    "What are the names of Developer Advocates? Use http://schema.org/ namespace for country name values.  Make rdfs properties optional",
    "What are the countries and names of persons living there in the data? Make rdfs properties optional",
    "Who can perform a demo to show Cloud App Programming? Return their names, IRI, and demo name.",
    "Who can present a event-enabling demo using Events Hub? Return only their names, IRI, and a demo name.",
    "What demo will show Cloud Application Programming and who can show it?",
][-1]

display(Markdown(f"## {prompt_from_user}"))

output = chain.invoke(input={"query": prompt_from_user})
display(Markdown(f"## {output['result']}"))

## What demo will show Cloud Application Programming and who can show it?



> Entering new HanaSparqlQAChain chain...
Generated SPARQL:
```sparql
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?eventName ?personName ?person

FROM <teched2025_devkeynote>
WHERE {
    ?event a schema:Event .
    ?event schema:name ?eventName .
    ?event schema:performer ?person .
    ?person a schema:Person .
    ?person schema:name ?personName .
    ?event schema:about ?service .
    ?service a schema:Service .
    ?service schema:name ?serviceName .
    FILTER(CONTAINS(UCASE(?serviceName), "SAP CLOUD APPLICATION PROGRAMMING MODEL"))
}
```
Final SPARQL:

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?eventName ?personName ?person

FROM <teched2025_devkeynote>
WHERE {
    ?event a schema:Event .
    ?event schema:name ?eventName .
    ?event schema:performer ?person .
    ?person a schema:Person .
    ?person schema:name ?personName .
    ?event sche

## The demo showcasing Cloud Application Programming is titled "SAP CAP MCP & AI Agents." The individuals who can present it are [Nico Schönteich](https://profile.sap.com/u/nicoschoenteich), [Thomas Jung](https://profile.sap.com/u/thomas_jung), and [DJ Adams](https://profile.sap.com/u/qmacro).